In [1]:
!pip install "pymongo[srv]"

In [ ]:
from pymongo import MongoClient
from dotenv import load_dotenv
from pathlib import Path
import os

# 프로젝트 루트의 .env 찾기
env_candidates = [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
]

for env_path in env_candidates:
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env 로드: {env_path}")
        break

uri = os.getenv("MONGODB_URI")

if not uri:
    raise RuntimeError("MONGODB_URI를 .env에서 찾을 수 없습니다.")

client = MongoClient(uri, serverSelectionTimeoutMS=10000)

try:
    client.admin.command("ping")
    print("MongoDB 연결 성공!")
except Exception as e:
    print("MongoDB 연결 실패:")
    print(e)
    raise

ConfigurationError: Empty host (or extra comma in host list)

In [ ]:
import pandas as pd

db = client["child_traffic_safety"]

main_df = pd.read_csv(
    "../data_processed/final_main_49.csv"
)

speed_df = pd.read_csv(
    "../data_processed/final_speed_bump_35.csv"
)

panel_229 = pd.read_csv(
    "../data_processed/panel_2020_2025_229.csv"
)

prediction_229 = pd.read_csv(
    "../results/predicted_high_risk_2026_229.csv"
)

blind_spots_49 = pd.read_csv(
    "../results/infrastructure_blind_spots_49.csv"
)

# 기존 값이 있으면 초기화

collections = {
    "final_main_49": main_df,
    "final_speed_bump_35": speed_df,
    "panel_2020_2025_229": panel_229,
    "prediction_2026_229": prediction_229,
    "infrastructure_blind_spots_49": blind_spots_49
}

for name, df in collections.items():
    db[name].delete_many({})
    db[name].insert_many(
        df.to_dict("records")
    )

print("MongoDB 최종 데이터 적재 완료!")

MongoDB 최종 데이터 적재 완료!


In [ ]:
print("final_main_49:",
      db["final_main_49"].count_documents({}))

print("final_speed_bump_35:",
      db["final_speed_bump_35"].count_documents({}))

print("panel_2020_2025_229:",
      db["panel_2020_2025_229"].count_documents({}))

print("prediction_2026_229:",
      db["prediction_2026_229"].count_documents({}))

print("infrastructure_blind_spots_49:",
      db["infrastructure_blind_spots_49"].count_documents({}))

final_main_49: 49
final_speed_bump_35: 35
panel_2020_2025_229: 1374
prediction_2026_229: 229
infrastructure_blind_spots_49: 49
